In [ ]:
# ═══════════════════════════════════════════════
# STEP 1: Install Ollama and pull Gemma 4
# Takes ~10 minutes on first run
# ═══════════════════════════════════════════════
import subprocess, sys, os, time, shutil

print("Installing zstd...")
subprocess.run("apt-get install -y zstd", shell=True, capture_output=True)

print("Installing Ollama...")
r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True, text=True)

# Install script exits non-zero in containers (systemd fails) but binary is still placed.
OLLAMA = shutil.which("ollama") or "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA):
    print("Install stdout:", r.stdout[-400:])
    print("Install stderr:", r.stderr[-400:])
    raise RuntimeError("Ollama binary not found after install")
print(f"Ollama installed: {OLLAMA}")

os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

print("Starting Ollama server...")
ollama_proc = subprocess.Popen(
    [OLLAMA, "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"},
)

for i in range(30):
    time.sleep(2)
    probe = subprocess.run(
        ["curl", "-s", "--max-time", "2", "http://localhost:11434/api/version"],
        capture_output=True
    )
    if probe.returncode == 0:
        print(f"Ollama server ready (PID {ollama_proc.pid}) after {(i+1)*2}s")
        break
else:
    raise RuntimeError("Ollama server did not start within 60 s")

print("Pulling Gemma 4 (~7 GB, ~8 min)...")
r = subprocess.run([OLLAMA, "pull", "gemma4:latest"],
                   capture_output=True, text=True)
if r.returncode == 0:
    print("Model ready.")
else:
    print("Pull stderr:", r.stderr[-500:])
    raise RuntimeError("gemma4:latest pull failed")


In [ ]:
# ═══════════════════════════════════════════════
# STEP 2: Install GuardianLens + Playwright
# ═══════════════════════════════════════════════
import urllib.request, zipfile

print("Downloading GuardianLens...")
urllib.request.urlretrieve(
    "https://github.com/natalia-jaskowska/guardianlens/archive/refs/heads/main.zip",
    "/tmp/gl.zip"
)
with zipfile.ZipFile("/tmp/gl.zip") as z:
    z.extractall("/tmp/")

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-input", "/tmp/guardianlens-main"],
    capture_output=True, text=True
)
print("GuardianLens installed." if r.returncode == 0 else f"Error: {r.stderr[-300:]}")
sys.path.insert(0, "/tmp/guardianlens-main/src")

print("Installing Playwright...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "playwright"],
               capture_output=True)
r = subprocess.run(["playwright", "install", "chromium", "--with-deps"],
                   capture_output=True, text=True)
print("Chromium ready." if r.returncode == 0 else f"Chromium note: {r.stderr[-200:]}")


In [ ]:
# ═══════════════════════════════════════════════
# STEP 3: Imports and helpers
# ═══════════════════════════════════════════════
from pathlib import Path
from IPython.display import display, HTML
import json, base64

SCREENSHOTS = Path("/kaggle/input/guardianlens-demo-screenshots")

THREAT_COLORS = {
    "safe":     ("#22c55e", "#dcfce7"),
    "caution":  ("#f59e0b", "#fef9c3"),
    "warning":  ("#f97316", "#fff7ed"),
    "alert":    ("#ef4444", "#fee2e2"),
    "critical": ("#7c3aed", "#ede9fe"),
}
THREAT_EMOJI = {"safe": "✅", "caution": "⚠️", "warning": "🔶", "alert": "🚨", "critical": "🆘"}

def img_b64(path):
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode()

def render_card(analysis):
    level = analysis["threat_level"]
    color, bg = THREAT_COLORS[level]
    emoji = THREAT_EMOJI[level]
    indicators = analysis.get("indicators", [])
    ind_html = "".join(
        f'<span style="background:{color};color:white;padding:3px 10px;border-radius:12px;margin:3px;font-size:13px;display:inline-block">{i}</span>'
        for i in indicators
    ) or '<span style="color:#888">none</span>'
    return f"""
    <div style="max-width:900px;margin:24px 0;font-family:sans-serif;border:2px solid {color};border-radius:12px;overflow:hidden">
      <div style="background:{color};padding:10px 16px;display:flex;align-items:center;gap:10px">
        <span style="font-size:26px">{emoji}</span>
        <span style="font-size:20px;font-weight:bold;color:white">{level.upper()}</span>
        <span style="font-size:14px;color:rgba(255,255,255,0.85);margin-left:4px">confidence: {analysis['confidence']:.0f}%</span>
        <span style="margin-left:auto;font-size:13px;color:rgba(255,255,255,0.75)">{analysis.get('platform','')}</span>
      </div>
      <img src="{img_b64(SCREENSHOTS / analysis['screenshot'])}" style="width:100%;display:block"/>
      <div style="background:{bg};padding:16px">
        <div style="margin-bottom:10px"><b>Category:</b> {analysis.get('category','—')} &nbsp;·&nbsp;
        <b>Grooming stage:</b> {analysis.get('grooming_stage','none')}</div>
        <div style="margin-bottom:10px"><b>Indicators:</b><br><div style="margin-top:4px">{ind_html}</div></div>
        <div style="margin-bottom:10px"><b>Narrative:</b><br><span style="color:#333;line-height:1.6">{analysis.get('narrative','')}</span></div>
        <div style="background:rgba(0,0,0,0.05);border-radius:6px;padding:10px;font-size:13px;color:#555;line-height:1.6">
          <b>Gemma 4 reasoning:</b><br>{analysis.get('reasoning','')}
        </div>
      </div>
    </div>"""

print("Ready.")


## Architecture

```
Child's device (client)            Parent's machine (server)
───────────────────────            ─────────────────────────────────────
guardlens-client                   FastAPI + Ollama (Gemma 4)
  mss screen capture  ──PNG──▶      Step 1: extract_conversations  (vision)
  httpx sender                      Step 2: match & merge history  (deterministic)
                                    Step 3: update_conv_status     (reasoning)
                                    Step 4: generate_parent_alert  (if threat ≥ WARNING)
                                    SQLite ← all state persisted after each step
                                    SSE dashboard :7860
```

**Gemma 4 features used:**
- **Multimodal vision** — reads screenshots directly, no OCR, works on any platform
- **Native function calling** — typed, Pydantic-validated outputs, no regex
- **Thinking chain** — full reasoning shown to parents, explainable AI
- **Runs via Ollama** — zero cloud, child's data never leaves the home

In [ ]:
# ═══════════════════════════════════════════════
# STEP 4: Prime the session with the safe TikTok DM
# ═══════════════════════════════════════════════
# Extract one representative frame from the recorded TikTok safe-chat
# MP4 (Emma + her friend Lily) and feed it through the real Gemma 4
# pipeline. The dashboard boots to a calm "everything's fine" baseline
# so the alert in Step 5 lands with maximum contrast.

import subprocess
from guardlens.pipeline import ConversationPipeline
from guardlens.config import OllamaConfig
from guardlens.database import GuardLensDatabase
from pathlib import Path
import json

DB_PATH = "/tmp/guardianlens_kaggle.db"
FRAMES  = Path("/tmp/frames"); FRAMES.mkdir(exist_ok=True)

ollama_cfg = OllamaConfig(host="http://localhost:11434", inference_model="gemma4:latest")
db         = GuardLensDatabase(Path(DB_PATH))
pipeline   = ConversationPipeline(ollama_cfg)

def extract_frame(video_path, out_path, timestamp):
    """Pull a single PNG from the MP4 at the given timestamp."""
    subprocess.run([
        "ffmpeg", "-y", "-ss", timestamp, "-i", str(video_path),
        "-frames:v", "1", "-q:v", "2", str(out_path),
    ], check=True, capture_output=True)
    return out_path

# Late-in-clip timestamps so the conversation is fully built up.
SAFE_FRAME = extract_frame(
    SCREENSHOTS / "demo_tiktok_safe.mp4",
    FRAMES / "tiktok_safe.png",
    "00:00:13",
)

results = []  # shared with Step 5's injector + Step 6's overview cards

def record_analysis(image_path):
    """Analyze one frame with Gemma 4 and append the DB result to results."""
    print(f"  {image_path.name}: ", end="", flush=True)
    conv_ids = pipeline.push_screenshot(image_path, db)
    if not conv_ids:
        print("no conversations extracted")
        results.append({"screenshot": image_path.name, "conv_id": None,
                        "platform": "Unknown", "status": {}})
        return
    for cid in conv_ids:
        row = db.get_conversation(cid)
        if not row:
            continue
        status = json.loads(row["status_json"]) if row["status_json"] else {}
        level  = status.get("threat_level", "safe")
        conf   = status.get("confidence", 0)
        print(f"{level.upper()} ({conf:.0f}%)")
        results.append({
            "screenshot": image_path.name,
            "conv_id":    cid,
            "platform":   row["platform"] or "Unknown",
            "status":     status,
        })

print("Priming baseline...")
record_analysis(SAFE_FRAME)
print(f"\nBaseline ready: {len(results)} conversation(s). Dashboard will open to a calm state.")


In [ ]:
# ═══════════════════════════════════════════════
# STEP 5: Live alert demo — TikTok grooming DM lands in real time
# ═══════════════════════════════════════════════
# Orchestration:
#   - Dashboard subprocess (FastAPI + SSE) starts on :7860 with the
#     safe-baseline DB from Step 4. Bell is grey; session is calm.
#   - Playwright opens the dashboard headless, records video, and
#     waits for a `.gl-toast` to appear when a new alert key lands in
#     the SSE snapshot.
#   - Meanwhile this kernel's injector thread extracts a representative
#     frame from the TikTok grooming DM (Emma + jake_r23) and pushes it
#     through real Gemma 4 inference. The pipeline writes the new
#     conversation to SQLite; the dashboard reads it on the next SSE
#     tick (every 2s); the browser fires the toast.
#
# Nothing here is precomputed — the toast, bell pulse, and detail
# drill-in are all driven by real model inference happening live while
# Playwright records.

import os, time, base64, threading, subprocess, sys
from pathlib import Path
from IPython.display import display, HTML

# Late-in-clip timestamp so the full grooming conversation (compliments,
# age probes, "you're really cute btw 😉", "i bet the guys at school
# love you", platform-switch ask) is visible in the frame.
ALERT_FRAME = extract_frame(
    SCREENSHOTS / "demo_tiktok_grooming.mp4",
    FRAMES / "tiktok_grooming.png",
    "00:00:31",
)
ALERT_FRAMES = [ALERT_FRAME]

# ── 1) Start the dashboard on the already-populated DB ───────────────
dash_env = {**os.environ,
            "GUARDLENS_DATABASE__PATH": DB_PATH,
            "GUARDLENS_OLLAMA__HOST": "http://localhost:11434"}

dash_proc = subprocess.Popen(
    [sys.executable, "/tmp/guardianlens-main/run.py",
     "--no-capture", "--dashboard-port", "7860", "--log-level", "WARNING"],
    env=dash_env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for i in range(25):
    time.sleep(2)
    if subprocess.run(["curl", "-s", "--max-time", "2", "http://localhost:7860/"],
                      capture_output=True).returncode == 0:
        print(f"Dashboard ready after {(i+1)*2}s")
        break
else:
    raise RuntimeError("Dashboard did not start")

SHOTS     = Path("/tmp/app_shots");  SHOTS.mkdir(exist_ok=True)
VIDEO_DIR = Path("/tmp/app_video");  VIDEO_DIR.mkdir(exist_ok=True)

# ── 2) Playwright observer script ────────────────────────────────────
observer_script = r"""
import asyncio, sys
from playwright.async_api import async_playwright
from pathlib import Path

SHOTS = Path("/tmp/app_shots")
N_ALERTS = int(sys.argv[1]) if len(sys.argv) > 1 else 1

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        context = await browser.new_context(
            viewport={"width": 1400, "height": 900},
            record_video_dir="/tmp/app_video/",
            record_video_size={"width": 1400, "height": 900},
        )
        page = await context.new_page()
        await page.goto("http://localhost:7860", wait_until="domcontentloaded", timeout=20000)

        await page.wait_for_function(
            "() => typeof ui !== 'undefined' && ui.knownAlerts instanceof Set",
            timeout=15000,
        )
        await page.screenshot(path=str(SHOTS / "00_clean.png"))
        print("READY", flush=True)

        for i in range(N_ALERTS):
            toast = page.locator(".gl-toast:not(.leaving)").first
            await toast.wait_for(state="visible", timeout=90000)
            await asyncio.sleep(1.4)
            await page.screenshot(path=str(SHOTS / f"{i+1:02d}_toast.png"))

            await toast.locator(".gl-toast-body").click()
            await asyncio.sleep(2.4)
            await page.screenshot(path=str(SHOTS / f"{i+1:02d}_detail_top.png"))

            await page.evaluate("document.getElementById('rightPanel').scrollTop = 600")
            await asyncio.sleep(1.8)
            await page.screenshot(path=str(SHOTS / f"{i+1:02d}_detail_reasoning.png"))

            back = page.locator("#stateConversation [data-back]").first
            await back.click()
            await asyncio.sleep(1.4)

        # Closing beat: open the bell menu so the final frame shows the
        # alert sitting in the inbox.
        await page.locator("#bellWrap").click()
        await asyncio.sleep(2.0)
        await page.screenshot(path=str(SHOTS / "99_bell_menu.png"))
        await asyncio.sleep(1.0)

        await context.close()
        await browser.close()

asyncio.run(main())
"""
Path("/tmp/take_shots.py").write_text(observer_script)

# ── 3) Launch Playwright and wait for its READY signal ───────────────
pw_proc = subprocess.Popen(
    [sys.executable, "/tmp/take_shots.py", str(len(ALERT_FRAMES))],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
)

ready = False
start = time.monotonic()
while time.monotonic() - start < 30:
    line = pw_proc.stdout.readline()
    if not line:
        break
    if line.strip() == "READY":
        ready = True
        break
if not ready:
    err = pw_proc.stderr.read() if pw_proc.stderr else ""
    print("Playwright failed to reach READY state")
    print(err[-500:])
    pw_proc.kill()
    raise RuntimeError("Playwright did not signal ready")

print("Observer ready — beginning injection sequence.")

# ── 4) Inject the grooming frame in parallel with Playwright ─────────
def inject_alerts():
    for path in ALERT_FRAMES:
        time.sleep(2.0)
        try:
            record_analysis(path)
        except Exception as exc:
            print(f"[inject] {path.name}: ERROR {exc}", flush=True)

injector = threading.Thread(target=inject_alerts, daemon=True)
injector.start()

# ── 5) Wait for both to finish ───────────────────────────────────────
stdout, stderr = pw_proc.communicate(timeout=300)
injector.join(timeout=60)
if pw_proc.returncode != 0:
    print("Playwright stderr:", (stderr or "")[-500:])

# ── 6) Encode the recording for inline playback ──────────────────────
webm_files = sorted(VIDEO_DIR.glob("*.webm"))
if webm_files:
    mp4_path = Path("/tmp/app_demo.mp4")
    subprocess.run([
        "ffmpeg", "-y", "-i", str(webm_files[0]),
        "-vf", "scale=1280:-2",
        "-c:v", "libx264", "-crf", "22", "-preset", "fast",
        "-movflags", "+faststart", "-pix_fmt", "yuv420p",
        str(mp4_path),
    ], capture_output=True, text=True)
    if mp4_path.exists():
        size_kb = mp4_path.stat().st_size // 1024
        mp4_b64 = base64.b64encode(mp4_path.read_bytes()).decode()
        print(f"Video: {size_kb} KB")
        display(HTML(f"""
<div style="max-width:960px;margin:16px 0;font-family:sans-serif">
  <div style="background:#0f172a;color:#f1f5f9;padding:12px 18px;border-radius:8px 8px 0 0;
              font-size:15px;font-weight:bold">
    GuardianLens Live — TikTok grooming DM detected in real time
    <span style="font-weight:normal;font-size:12px;color:#94a3b8;margin-left:10px">
      Playwright recording · Kaggle T4 · real Gemma 4 inference
    </span>
  </div>
  <video controls autoplay loop muted
         style="width:100%;display:block;border-radius:0 0 8px 8px;background:#000">
    <source src="data:video/mp4;base64,{mp4_b64}" type="video/mp4"/>
  </video>
</div>"""))
    else:
        print("MP4 encode failed — falling back to screenshots")
        for fname in sorted(SHOTS.glob("*.png")):
            b64 = "data:image/png;base64," + base64.b64encode(fname.read_bytes()).decode()
            display(HTML(f'<img src="{b64}" style="width:100%;margin:4px 0;border-radius:6px"/>'))
else:
    print("No WebM recorded — showing screenshots")
    for fname in sorted(SHOTS.glob("*.png")):
        b64 = "data:image/png;base64," + base64.b64encode(fname.read_bytes()).decode()
        display(HTML(f'<img src="{b64}" style="width:100%;margin:4px 0;border-radius:6px"/>'))


In [ ]:
# ═══════════════════════════════════════════════
# STEP 6: Alert overview — one card per flagged conversation
# ═══════════════════════════════════════════════
FLAGGED_LEVELS = {"warning", "alert", "critical"}

flagged = [r for r in results if r["status"].get("threat_level") in FLAGGED_LEVELS]
print(f"{len(flagged)} flagged conversation(s) out of {len(results)} analyzed\n")

for item in flagged:
    s = item["status"]
    display(HTML(render_card({
        "screenshot":    item["screenshot"],
        "platform":      item["platform"],
        "threat_level":  s.get("threat_level", "warning"),
        "category":      s.get("category", ""),
        "confidence":    s.get("confidence", 0),
        "grooming_stage": s.get("grooming_stage", "none"),
        "indicators":    s.get("indicators", []),
        "narrative":     s.get("narrative", ""),
        "reasoning":     s.get("reasoning", ""),
    })))


In [ ]:
# ═══════════════════════════════════════════════
# STEP 7: Gemma 4's thinking chain — the TikTok grooming DM, unrolled
# ═══════════════════════════════════════════════
# pipeline._analyze_frame is the vision step that reads the screenshot
# and extracts chats. It returns both the structured function-call
# output and the raw thinking trace the model produced on the way there.

TARGET = FRAMES / "tiktok_grooming.png"
print(f"Re-running frame analysis on {TARGET.name} to capture the thinking chain...")
frame_result = pipeline._analyze_frame(TARGET)
thinking = frame_result.raw_thinking or "(thinking chain not captured for this result)"
display(HTML(f"""
<div style="background:#1e1e2e;color:#cdd6f4;border-radius:8px;padding:20px;
            font-family:monospace;font-size:13px;line-height:1.6;max-width:900px">
  <div style="color:#89b4fa;font-size:15px;margin-bottom:12px">
    Gemma 4 thinking chain — {TARGET.name}
  </div>
  <pre style="white-space:pre-wrap;margin:0;color:#cdd6f4">{thinking[:1500]}</pre>
</div>
"""))


In [ ]:
from guardlens.schema import ConversationStatus
print("Pydantic model that validates every Gemma 4 response (zero raw dicts):\n")
print(json.dumps(ConversationStatus.model_json_schema(), indent=2))